# Formelsamling — SW2PLA
Komplet oversigt over alle formler med forklaringer og kodeeksempler.

**Indhold:**
1. [Vektorer](#vektorer) — norm, enhedsvektor, dot product, lineær uafhængighed
2. [Matricer](#matricer) — oprettelse, trace, determinant, invers, norm
3. [Vektorrum](#vektorrum) — rang, nullrum, søjlerum
4. [Ligningssystemer](#ligningssystemer) — np.linalg.solve, rækkereduktion (RREF)
5. [LU-dekomposition](#lu)
6. [QR-dekomposition](#qr)
7. [Mindste kvadrater / GLM](#glm)
8. [Egendekomposition](#eigen)
9. [SVD](#svd)
10. [Low-rank approksimation](#lowrank)
11. [PCA](#pca)


In [24]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt
from sympy import Matrix as symMatrix


---
## 1. Vektorer <a id='vektorer'></a>

### 1.1 Norm af en vektor
Normen (L2/euklidisk) af en vektor måler dens længde:

$$\|v\|_2 = \sqrt{\sum_{i=1}^n v_i^2}$$

**Eksempel:** Givet $v = [4, 3]$, beregn normen.


In [25]:
v = np.array([4, 3])

norm_v = np.linalg.norm(v)
print("Norm af v:", norm_v)


Norm af v: 5.0


### 1.2 Enhedsvektor
En enhedsvektor peger i samme retning som $v$, men har længde 1:

$$\hat{v} = \frac{v}{\|v\|}$$


In [26]:
unit_v = v / norm_v
print("Enhedsvektor:", unit_v)
print("Verifikation — norm af enhedsvektor:", np.linalg.norm(unit_v))


Enhedsvektor: [0.8 0.6]
Verifikation — norm af enhedsvektor: 1.0


### 1.3 Dot product (indre produkt)
$$v \cdot w = \sum_{i=1}^n v_i w_i$$

Hvis $v \cdot w = 0$ er vektorerne **ortogonale** (vinkelrette).


In [27]:
v = np.array([4, 3])
w = np.array([5, 2])

dot = np.dot(v, w)
print("Dot product v · w:", dot)

# Tjek ortogonalitet
v1 = np.array([7, 0, 0])
v2 = np.array([0, 1, 2])
print("\nDot product v1 · v2:", np.dot(v1, v2), "→ ortogonale:", np.dot(v1, v2) == 0)


Dot product v · w: 26

Dot product v1 · v2: 0 → ortogonale: True


### 1.4 Lineær uafhængighed via determinant
Vektorer er **lineært uafhængige** hvis determinanten af matricen de danner ≠ 0.

$$\det(A) \neq 0 \implies \text{søjlerne er lineært uafhængige}$$


In [28]:
# 2 vektorer i R2
v1 = np.array([1, 2])
v2 = np.array([3, 4])
A = np.column_stack([v1, v2])

det = np.linalg.det(A)
print("Determinant:", round(det, 6))
print("Lineært uafhængige:", abs(det) > 1e-10)


Determinant: -2.0
Lineært uafhængige: True


---
## 2. Matricer <a id='matricer'></a>

### 2.1 Oprettelse og basale operationer


In [29]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

print("Matrix A:")
print(A)
print("\nShape:", A.shape)
print("Transpose A.T:")
print(A.T)


Matrix A:
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]

Shape: (3, 3)
Transpose A.T:
[[1. 4. 7.]
 [2. 5. 8.]
 [3. 6. 9.]]


### 2.2 Trace
Sporet (trace) er summen af diagonalelementerne. Kun defineret for kvadratiske matricer:

$$\text{tr}(A) = \sum_{i=1}^n a_{ii}$$


In [30]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

trace = np.trace(A)
print("Trace af A:", trace)


Trace af A: 15.0


### 2.3 Determinant
$$\det(A) \neq 0 \implies A \text{ er invertibel (fuld rang, lin. uafh. søjler)}$$


In [31]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

det = np.linalg.det(A)
print("Determinant af A:", round(det, 6))
print("Invertibel:", abs(det) > 1e-10)


Determinant af A: 0.0
Invertibel: False


### 2.4 Matrix-invers
For en kvadratisk matrix $A$ gælder: $A \cdot A^{-1} = I$

En matrix er invertibel hvis og kun hvis:
- $\det(A) \neq 0$
- $A$ har fuld rang
- Søjlerne er lineært uafhængige


In [32]:
A = np.array([[2, 1, 0],
              [4, 3, 2],
              [1, 0, 5]], dtype=float)

A_inv = np.linalg.inv(A)
print("A_inv:")
print(np.round(A_inv, 4))

# Verifikation: A @ A_inv skal give identitetsmatrix
print("\nVerifikation A @ A_inv:")
print(np.round(A @ A_inv, 6))


A_inv:
[[ 1.25   -0.4167  0.1667]
 [-1.5     0.8333 -0.3333]
 [-0.25    0.0833  0.1667]]

Verifikation A @ A_inv:
[[ 1. -0.  0.]
 [ 0.  1.  0.]
 [ 0.  0.  1.]]


### 2.5 Matrix norm (Frobenius)
Frobenius-normen udvider vektornormen til matricer:

$$\|A\|_F = \sqrt{\sum_{i,j} a_{ij}^2}$$


In [33]:
A = np.array([[1, 2],
              [3, 4]], dtype=float)

norm_A = np.linalg.norm(A)          # Frobenius (default)
print("Frobenius norm af A:", norm_A)


Frobenius norm af A: 5.477225575051661


### 2.6 Matrixmultiplikation
Brug `@`-operatoren. Indre dimensioner skal passe: $(m \times k) \cdot (k \times n) = (m \times n)$


In [34]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

C = A @ B
print("A @ B =")
print(C)


A @ B =
[[19 22]
 [43 50]]


---
## 3. Rang, nullrum & lineær uafhængighed <a id='vektorrum'></a>

### 3.1 Rang
Rangen er antallet af lineært uafhængige søjler (= dimensionen af søjlerummet):

$$\text{rank}(A) = \dim(\text{column space})$$


In [35]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

rank = np.linalg.matrix_rank(A)
print("Rang af A:", rank)
print("A har fuld rang:", rank == min(A.shape))


Rang af A: 2
A har fuld rang: False


### 3.2 Nullrum og rang-nullitets-sætningen
$$\text{antal søjler} = \text{rank}(A) + \dim(\text{nullspace}(A))$$

Et vector $v$ tilhører nullrummet hvis $Av = 0$.


In [36]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

rank = np.linalg.matrix_rank(A)
num_cols = A.shape[1]
nullity = num_cols - rank

print("Antal søjler:", num_cols)
print("Rang:", rank)
print("Dim(nullspace) =", nullity)

# Test om en vektor tilhører nullrummet
v = np.array([1, -2, 1])
result = A @ v
print("\nAv =", np.round(result, 10), "→ v tilhører nullspace:", np.allclose(result, 0))


Antal søjler: 3
Rang: 2
Dim(nullspace) = 1

Av = [0. 0. 0.] → v tilhører nullspace: True


---
## 4. Løsning af lineære ligningssystemer <a id='ligningssystemer'></a>

### 4.1 np.linalg.solve
Løser $Ax = b$ for kvadratisk $A$ med unik løsning:

$$Ax = b \implies x = A^{-1}b$$

**Eksempel:**
$$\begin{cases} 2x + 2y = 8 \\ -x + 2y = 4 \end{cases}$$


In [37]:
A = np.array([[ 2,  2],
              [-1,  2]], dtype=float)
b = np.array([8, 4], dtype=float)

x = np.linalg.solve(A, b)
print("Løsning x =", x[0], ", y =", x[1])

# Verifikation
print("Verifikation A @ x =", A @ x, "(skal være", b, ")")


Løsning x = 1.3333333333333335 , y = 2.6666666666666665
Verifikation A @ x = [8. 4.] (skal være [8. 4.] )


### 4.2 Rækkereduktion (RREF) med sympy
Bruges til at analysere løsninger og finde antal løsninger:
- $\text{rank}([A|b]) = \text{rank}(A)$ → løsning eksisterer
- $\text{rank}(A) = n$ → unik løsning
- $\text{rank}(A) < n$ → uendeligt mange løsninger


In [38]:
from sympy import Matrix

A = np.array([[ 2,  2],
              [-1,  2]], dtype=float)
b = np.array([[8], [4]], dtype=float)

# Augmented matrix [A|b]
Ab = np.hstack([A, b])
rref, pivot_cols = Matrix(Ab).rref()
print("RREF af [A|b]:")
print(rref)


RREF af [A|b]:
Matrix([[1, 0, 1.33333333333333], [0, 1, 2.66666666666667]])


---
## 5. LU-dekomposition <a id='lu'></a>

$$A = PLU$$

- $P$ = permutationsmatrix (ombytning af rækker)
- $L$ = nedre triangulær matrix (lower)
- $U$ = øvre triangulær matrix (upper)

**Bruges til:** løsning af ligningssystemer, determinantberegning.


In [39]:
import scipy.linalg

A = np.array([[1, 2, 3],
              [0, 1, 4],
              [2, 4, 10]], dtype=float)

P, L, U = scipy.linalg.lu(A)

print("P (permutation):")
print(P)
print("\nL (lower triangular):")
print(np.round(L, 4))
print("\nU (upper triangular):")
print(np.round(U, 4))

# Verifikation: P @ L @ U skal give A
print("\nVerifikation P @ L @ U:")
print(np.round(P @ L @ U, 6))


P (permutation):
[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]

L (lower triangular):
[[1.  0.  0. ]
 [0.  1.  0. ]
 [0.5 0.  1. ]]

U (upper triangular):
[[ 2.  4. 10.]
 [ 0.  1.  4.]
 [ 0.  0. -2.]]

Verifikation P @ L @ U:
[[ 1.  2.  3.]
 [ 0.  1.  4.]
 [ 2.  4. 10.]]


---
## 6. QR-dekomposition <a id='qr'></a>

$$A = QR$$

- $Q$ = ortogonal matrix ($Q^T Q = I$, $Q^{-1} = Q^T$)
- $R$ = øvre triangulær matrix

**Ortogonal matrix egenskaber:**
- $Q^{-1} = Q^T$
- $\det(Q) = \pm 1$
- $Q$ er altid invertibel
- $Q^T Q = I$ (hurtig tjek)


In [40]:
A = np.array([[1, 2, 3],
              [0, 1, 4],
              [5, 6, 0]], dtype=float)

Q, R = np.linalg.qr(A)

print("Q (ortogonal matrix):")
print(np.round(Q, 4))
print("\nR (øvre triangulær):")
print(np.round(R, 4))

# Tjek Q er ortogonal: Q.T @ Q = I
print("\nTjek Q.T @ Q = I:")
print(np.round(Q.T @ Q, 6))

# Verifikation: Q @ R = A
print("\nVerifikation Q @ R:")
print(np.round(Q @ R, 6))


Q (ortogonal matrix):
[[-0.1961 -0.6052 -0.7715]
 [-0.     -0.7868  0.6172]
 [-0.9806  0.121   0.1543]]

R (øvre triangulær):
[[-5.099  -6.2757 -0.5883]
 [ 0.     -1.271  -4.9629]
 [ 0.      0.      0.1543]]

Tjek Q.T @ Q = I:
[[ 1. -0. -0.]
 [-0.  1. -0.]
 [-0. -0.  1.]]

Verifikation Q @ R:
[[ 1.  2.  3.]
 [ 0.  1.  4.]
 [ 5.  6. -0.]]


---
## 7. Generalized Linear Model (GLM) & Mindste Kvadrater <a id='glm'></a>

**Modellen:** $y = \beta_1 x + \beta_0$

**De 4 trin:**
1. Definer designmatrix $X$ (kolonner: $[x, 1]$)
2. Løs $\beta = (X^T X)^{-1} X^T y$ med `np.linalg.lstsq`
3. Beregn forudsigelser $\hat{y} = X\beta$
4. Beregn SSE: $\text{SSE} = \sum(y - \hat{y})^2$

**Normalligningen:** $X^T X \beta = X^T y$


In [41]:
# Data
hours  = np.array([3, 5, 6, 8, 9, 4, 7], dtype=float)
scores = np.array([55, 70, 75, 88, 90, 62, 82], dtype=float)

# Trin 1: Designmatrix [x, 1]
X = np.column_stack([hours, np.ones(len(hours))])

# Trin 2: Løs med mindste kvadrater
beta, residuals, rank, sv = np.linalg.lstsq(X, scores, rcond=None)
beta1, beta0 = beta
print(f"Model: y = {beta1:.4f}x + {beta0:.4f}")

# Trin 3: Forudsigelse
x_new = 6.5
y_pred = beta1 * x_new + beta0
print(f"Forudsagt score for {x_new} timer: {y_pred:.2f}")

# Trin 4: SSE
y_hat = X @ beta
SSE = np.sum((scores - y_hat)**2)
print(f"SSE: {SSE:.4f}")


Model: y = 6.0357x + 38.3571
Forudsagt score for 6.5 timer: 77.59
SSE: 15.6786


---
## 8. Egendekomposition (Diagonalisering) <a id='eigen'></a>

$$Av = \lambda v$$

- $\lambda$ = egenværdi
- $v$ = egenvektor

**Diagonalisering:**
$$A = V \Lambda V^{-1}$$

- $V$ = matrix med egenvektorer som søjler
- $\Lambda$ = diagonalmatrix med egenværdier

**Spektralsætningen:** Symmetrisk $A$ har reelle egenværdier og ortogonale egenvektorer.


In [42]:
B = np.array([[4,  0,  0],
              [1,  2,  0],
              [-3, 0,  3]], dtype=float)

# Egendekomposition
eigenvalues, eigenvectors = np.linalg.eig(B)

print("Egenværdier (lambda):")
print(np.round(eigenvalues, 4))

V = eigenvectors
D = np.diag(eigenvalues)
V_inv = np.linalg.inv(V)

print("\nV (egenvektorer som søjler):")
print(np.round(V, 4))
print("\nD (diagonal med egenværdier):")
print(np.round(D, 4))

# Rekonstruktion og verifikation
B_reconstructed = V @ D @ V_inv
print("\nVerifikation V @ D @ V_inv ≈ B:")
print(np.round(B_reconstructed, 6))


Egenværdier (lambda):
[3. 2. 4.]

V (egenvektorer som søjler):
[[ 0.      0.      0.3123]
 [ 0.      1.      0.1562]
 [ 1.      0.     -0.937 ]]

D (diagonal med egenværdier):
[[3. 0. 0.]
 [0. 2. 0.]
 [0. 0. 4.]]

Verifikation V @ D @ V_inv ≈ B:
[[ 4.  0.  0.]
 [ 1.  2.  0.]
 [-3.  0.  3.]]


---
## 9. Singular Value Decomposition (SVD) <a id='svd'></a>

$$A = U \Sigma V^T$$

- $U$ ∈ $\mathbb{R}^{m \times m}$ = venstre singulære vektorer (ortogonal)
- $\Sigma$ ∈ $\mathbb{R}^{m \times n}$ = singulærværdier på diagonalen ($\sigma_1 \geq \sigma_2 \geq \ldots > 0$)
- $V^T$ ∈ $\mathbb{R}^{n \times n}$ = højre singulære vektorer (ortogonal)

**Relation til egendekomposition:**
- $A^T A = V \Sigma^2 V^T$ → $V$ og $\sigma_i^2$ er egenværdier af $A^T A$
- $A A^T = U \Sigma^2 U^T$ → $U$ er egenvektorer af $A A^T$


In [43]:
B = np.array([[4,  0,  0],
              [1,  2,  0],
              [-3, 0,  3]], dtype=float)

# SVD
U, S, Vt = np.linalg.svd(B)

print("U:")
print(np.round(U, 4))
print("\nS (singulærværdier):")
print(np.round(S, 4))
print("\nVt:")
print(np.round(Vt, 4))

# Rekonstruktion: U @ diag(S) @ Vt
Sigma = np.zeros_like(B)
np.fill_diagonal(Sigma, S)

B_reconstructed = U @ Sigma @ Vt
print("\nVerifikation U @ Sigma @ Vt ≈ B:")
print(np.round(B_reconstructed, 6))


U:
[[-0.6695  0.5564 -0.4921]
 [-0.1931  0.5094  0.8386]
 [ 0.7172  0.6565 -0.2336]]

S (singulærværdier):
[5.4781 2.3457 1.8677]

Vt:
[[-0.9169 -0.0705  0.3928]
 [ 0.3263  0.4343  0.8396]
 [-0.2298  0.898  -0.3752]]

Verifikation U @ Sigma @ Vt ≈ B:
[[ 4.  0. -0.]
 [ 1.  2.  0.]
 [-3.  0.  3.]]


---
## 10. Low-rank approksimation <a id='lowrank'></a>

Rekonstruktionen af $A$ som sum af rank-1 lag:

$$A = \sum_{i=1}^{r} \sigma_i \, u_i v_i^T$$

En **low-rank approksimation** med $k < r$ led:

$$A \approx \hat{A}_k = \sum_{i=1}^{k} \sigma_i \, u_i v_i^T$$

Jo større $\sigma_i$, jo mere information bidrager det led.  
Ved at beholde de $k$ største singulærværdier bevares det meste af informationen.


In [44]:
B = np.array([[4,  0,  0],
              [1,  2,  0],
              [-3, 0,  3]], dtype=float)

U, S, Vt = np.linalg.svd(B)

# Fuld rekonstruktion (alle rank-1 lag)
B_full = sum(S[i] * np.outer(U[:, i], Vt[i, :]) for i in range(len(S)))
print("Fuld rekonstruktion (r lag):")
print(np.round(B_full, 4))

# Low-rank approksimation med k=2
k = 2
B_approx = sum(S[i] * np.outer(U[:, i], Vt[i, :]) for i in range(k))
print(f"\nLow-rank approksimation (k={k} lag):")
print(np.round(B_approx, 4))

print(f"\nFejl (Frobenius norm): {np.linalg.norm(B - B_approx):.4f}")


Fuld rekonstruktion (r lag):
[[ 4.  0. -0.]
 [ 1.  2.  0.]
 [-3.  0.  3.]]

Low-rank approksimation (k=2 lag):
[[ 3.7888  0.8254 -0.3449]
 [ 1.3599  0.5935  0.5877]
 [-3.1003  0.3918  2.8363]]

Fejl (Frobenius norm): 1.8677


---
## 11. PCA — Principal Component Analysis <a id='pca'></a>

**De 5 trin:**
1. **Mean-center** data: $X \leftarrow X - \bar{X}$
2. **Kovariansmatrix:** $C = \frac{X^T X}{n-1}$
3. **Egendekomposition** af $C$: egenværdier $\lambda_i$ og egenvektorer
4. **Sortér** efter faldende egenværdier
5. **Projicér** data: $\text{components} = X \cdot V_k$

**Forklaret varians:** $\frac{\lambda_i}{\sum \lambda_j} \times 100\%$


In [45]:
np.random.seed(42)
X = np.random.randn(100, 3)
X[:, 1] += X[:, 0]  # lav korrelation

# Trin 1: Mean-center
X = X - np.mean(X, axis=0, keepdims=True)

# Trin 2: Kovariansmatrix
C = X.T @ X / (X.shape[0] - 1)
print("Kovariansmatrix C:")
print(np.round(C, 4))

# Trin 3: Egendekomposition
evals, evecs = np.linalg.eigh(C)

# Trin 4: Sortér faldende
idx = np.argsort(evals)[::-1]
evals = evals[idx]
evecs = evecs[:, idx]

# Trin 5: Projicér (behold 2 komponenter)
components = X @ evecs[:, :2]
print("\nShape af projicerede data:", components.shape)

# Forklaret varians
var_explained = evals / np.sum(evals) * 100
print("\nForklaret varians per komponent (%):")
for i, v in enumerate(var_explained):
    print(f"  PC{i+1}: {v:.1f}%")


Kovariansmatrix C:
[[ 0.6803  0.641  -0.1108]
 [ 0.641   1.5604 -0.2457]
 [-0.1108 -0.2457  1.2386]]

Shape af projicerede data: (100, 2)

Forklaret varians per komponent (%):
  PC1: 57.3%
  PC2: 32.9%
  PC3: 9.8%


---
## Hurtig oversigt — Python-funktioner

| Opgave | Kode | Hvad fortæller det dig? |
|---|---|---|
| Norm af vektor | `np.linalg.norm(v)` | Vektorens **længde** i rummet. Stor norm = lang vektor; lille norm = kort vektor. |
| Enhedsvektor | `v / np.linalg.norm(v)` | Isolerer **retningen** — fjerner størrelsen så normen = 1. |
| Dot product | `np.dot(v, w)` | Måler hvor meget to vektorer **peger i samme retning**. Resultat = 0 → vinkelrette (ortogonale). |
| Matrixmultiplikation | `A @ B` | Sammensætter to lineære transformationer. Kræver at indre dimensioner matcher: (m×k)·(k×n). |
| Transpose | `A.T` | Spejler matricen om diagonalen — søjler bliver rækker og omvendt. |
| Trace | `np.trace(A)` | Summen af diagonalelementerne. Er **lig summen af egenværdierne** — et mål for matrixens "total strækning". |
| Determinant | `np.linalg.det(A)` | Fortæller om matricen er **invertibel**: ≠ 0 → invertibel, = 0 → singulær (søjlerne er lin. afhængige). |
| Invers | `np.linalg.inv(A)` | "Fortryder" en matrixtransformation: A · A⁻¹ = I. Eksisterer kun når det(A) ≠ 0. |
| Rang | `np.linalg.matrix_rank(A)` | Antal **lineært uafhængige søjler**. Fortæller dimensionen af søjlerummet og om systemet har unik løsning. |
| Frobenius norm | `np.linalg.norm(A)` | Den samlede "størrelse" af en matrix — roden af summen af alle elementers kvadrat. |
| Løs Ax=b | `np.linalg.solve(A, b)` | Find x der opfylder ligningssystemet eksakt. Kræver kvadratisk A med fuld rang (unik løsning). |
| Mindste kvadrater | `np.linalg.lstsq(X, y, rcond=None)` | Find den bedste approksimation når Ax=b **ikke** har eksakt løsning. Minimerer den samlede fejl (SSE). |
| LU-dekomposition | `scipy.linalg.lu(A)` → P, L, U | Nedbryder A til trekantmatricer for hurtig løsning af ligningssystemer og determinantberegning. |
| QR-dekomposition | `np.linalg.qr(A)` → Q, R | Q er ortogonal (bevarer længder og vinkler), R er øvre trekant. Bruges i mindste kvadrater og numeriske metoder. |
| Egendekomposition | `np.linalg.eig(A)` → egenværdier, egenvektorer | Egenværdi λ: "hvor meget strækkes rummet i denne retning?". Egenvektor v: "hvilken retning ændres ikke af A?". |
| SVD | `np.linalg.svd(A)` → U, S, Vt | Generaliserer egendekomposition til alle matricer. Singulærværdierne rangordner hvor meget information hvert lag indeholder. |
| RREF | `sympy.Matrix(A).rref()` | Forenkler et ligningssystem til enkleste form. Afslører rang, pivotkolonner og om der er 0, 1 eller ∞ løsninger. |